# Kinematische Analyse des Kuka KR6 900-2 agilius

<figure>
<center>
<img width=500 src='figures/kr6.png' />
<figcaption>Kuka KR 6 900-2</figcaption></center>
</figure>

Die Denavit-Hartenberg Parameter ergeben sich wie folgt:
<!--
\begin{array}{c|c|c|c|c}
  i & \theta_i & d_i & a_i & \alpha_i \\
  \hline
			1 & \theta_1 & 0,08920 & 0 & +90^\circ\\
			2 & \theta_2 & 0 & 0,425 & 0\\
			3 & \theta_3 & 0 & 0,392 & 0\\
			4 & \theta_4 & 0,10930 & 0 & -90^\circ\\
			5 & \theta_5 & 0,09475 & 0 & +90^\circ\\
			6 & \theta_6 & 0,08250 & 0 & 0
\end{array}
-->

\begin{array}{c|c|c|c|c}
  i & \theta_i & d_i & a_i & \alpha_i \\
  \hline
			1 & \theta_1 & d_1 & a_1 & +90^\circ\\
			2 & \theta_2 & 0 & a_2 & 0\\
			3 & \theta_3 & 0 & a_3 & +90^\circ\\
			4 & \theta_4 & d_4 & 0 & -90^\circ\\
			5 & \theta_5 & 0 & 0 & +90^\circ\\
			6 & \theta_6 & d_6 & 0 & 0
\end{array}

mit $d_1 = 0.400, a_1 = 0.025, a_2 = 0.455, a_3 = 0.025, d_4 = 0.420, d_6 = 0,090$

Die DH-Matrizen ergeben sich wie folgt:

In [1]:
from sympy import *
from IPython.display import display, Latex
init_printing(use_latex='mathjax')

In [2]:
def dhFrame(theta, d, a, alpha):
    
    rot_theta = Matrix([ [cos(theta), -sin(theta), 0, 0], 
                         [sin(theta), cos(theta),  0, 0], 
                         [0,             0,        1, 0], 
                         [0,             0,        0, 1] ])
    
    trans_d = Matrix([ [1, 0, 0, 0],
                       [0, 1, 0, 0],
                       [0, 0, 1, d],
                       [0, 0, 0, 1] ])
    
    trans_a = Matrix([ [1, 0, 0, a], 
                       [0, 1, 0, 0], 
                       [0, 0, 1, 0], 
                       [0, 0, 0, 1] ])
    
    rot_alpha = Matrix([ [1,          0,           0, 0], 
                         [0, cos(alpha), -sin(alpha), 0], 
                         [0, sin(alpha),  cos(alpha), 0], 
                         [0,          0,           0, 1] ])
    
    dh_frame = rot_theta * trans_d * trans_a * rot_alpha
    
    return dh_frame;

Wir lassen uns zuerst die allgemeine DH-Matrix $^{i-1}\mathbf{T}_i$ ausgeben:

In [3]:
theta_i, alpha_i, a_i, d_i = symbols('theta_i alpha_i a_i d_i')
Ti = symbols('{}^{i-1}\mathbf{T}_i')

Tdh = dhFrame(theta_i, d_i, a_i, alpha_i)

display(Ti, Tdh)

{}_i__{i-1}\mathbf{T}

⎡cos(θᵢ)  -sin(θᵢ)⋅cos(αᵢ)  sin(αᵢ)⋅sin(θᵢ)   aᵢ⋅cos(θᵢ)⎤
⎢                                                       ⎥
⎢sin(θᵢ)  cos(αᵢ)⋅cos(θᵢ)   -sin(αᵢ)⋅cos(θᵢ)  aᵢ⋅sin(θᵢ)⎥
⎢                                                       ⎥
⎢   0         sin(αᵢ)           cos(αᵢ)           dᵢ    ⎥
⎢                                                       ⎥
⎣   0            0                 0              1     ⎦

Danach werden die Matrizen $^{0}\mathbf{T}_1$ bis $^{5}\mathbf{T}_6$ berechnet.

In [5]:
Ts = symbols('{}^0:7\mathbf{T}_0:7')

T = zeros(7,7)
for i in range(0, 7):
  for j in range(0, 7):
    if j+i > 6:
      break
    T[i,j+i] = Ts[8 * i + j]

T0 = ones(1,7)
T0[1] = T[0,1]
for i in range(2, 7):
  T0[i] = T0[i-1] * T[i-1,i]

display(T[0,6], T0[6])

{}_6__0\mathbf{T}

{}_1__0\mathbf{T}⋅{}_2__1\mathbf{T}⋅{}_3__2\mathbf{T}⋅{}_4__3\mathbf{T}⋅{}_5__ ↪

↪ 4\mathbf{T}⋅{}_6__5\mathbf{T}

In [6]:
theta = symbols('theta_1:7')
d =  symbols('d_1:7')
a =  symbols('a_1:7')
T01 = dhFrame(theta[0], d[0], a[0], pi/2)

display(T[0,1], T01)

{}_1__0\mathbf{T}

⎡cos(θ₁)  0  sin(θ₁)   a₁⋅cos(θ₁)⎤
⎢                                ⎥
⎢sin(θ₁)  0  -cos(θ₁)  a₁⋅sin(θ₁)⎥
⎢                                ⎥
⎢   0     1     0          d₁    ⎥
⎢                                ⎥
⎣   0     0     0          1     ⎦

In [7]:
T12 = dhFrame(theta[1], 0, a[1], 0)

display(T[1,2], T12)

{}_2__1\mathbf{T}

⎡cos(θ₂)  -sin(θ₂)  0  a₂⋅cos(θ₂)⎤
⎢                                ⎥
⎢sin(θ₂)  cos(θ₂)   0  a₂⋅sin(θ₂)⎥
⎢                                ⎥
⎢   0        0      1      0     ⎥
⎢                                ⎥
⎣   0        0      0      1     ⎦

In [8]:
T23 = dhFrame(theta[2], 0, a[2], +pi/2)

display(T[2,3], T23)

{}_3__2\mathbf{T}

⎡cos(θ₃)  0  sin(θ₃)   a₃⋅cos(θ₃)⎤
⎢                                ⎥
⎢sin(θ₃)  0  -cos(θ₃)  a₃⋅sin(θ₃)⎥
⎢                                ⎥
⎢   0     1     0          0     ⎥
⎢                                ⎥
⎣   0     0     0          1     ⎦

In [9]:
T34 = dhFrame(theta[3], d[3], 0, -pi/2)

display(T[3,4], T34)

{}_4__3\mathbf{T}

⎡cos(θ₄)  0   -sin(θ₄)  0 ⎤
⎢                         ⎥
⎢sin(θ₄)  0   cos(θ₄)   0 ⎥
⎢                         ⎥
⎢   0     -1     0      d₄⎥
⎢                         ⎥
⎣   0     0      0      1 ⎦

In [10]:
T45 = dhFrame(theta[4], 0, 0, pi/2)

display(T[4,5], T45)

{}_5__4\mathbf{T}

⎡cos(θ₅)  0  sin(θ₅)   0⎤
⎢                       ⎥
⎢sin(θ₅)  0  -cos(θ₅)  0⎥
⎢                       ⎥
⎢   0     1     0      0⎥
⎢                       ⎥
⎣   0     0     0      1⎦

In [11]:
T56 = dhFrame(theta[5], d[5], 0, 0)

display(T[5,6], T56)

{}_6__5\mathbf{T}

⎡cos(θ₆)  -sin(θ₆)  0  0 ⎤
⎢                        ⎥
⎢sin(θ₆)  cos(θ₆)   0  0 ⎥
⎢                        ⎥
⎢   0        0      1  d₆⎥
⎢                        ⎥
⎣   0        0      0  1 ⎦

Die gesamte Transformation von $K_0$ bis $K_6$ ergibt sich aus der Multipikaltion

In [12]:
T06 = T01 * T12 * T23 * T34 * T45 * T56

#display_result(T[0,6], T0[6], T06)
display(T06)

⎡(((-sin(θ₂)⋅sin(θ₃)⋅cos(θ₁) + cos(θ₁)⋅cos(θ₂)⋅cos(θ₃))⋅cos(θ₄) + sin(θ₁)⋅sin( ↪
⎢                                                                              ↪
⎢(((-sin(θ₁)⋅sin(θ₂)⋅sin(θ₃) + sin(θ₁)⋅cos(θ₂)⋅cos(θ₃))⋅cos(θ₄) - sin(θ₄)⋅cos( ↪
⎢                                                                              ↪
⎢                                             ((-sin(θ₂)⋅sin(θ₃) + cos(θ₂)⋅cos ↪
⎢                                                                              ↪
⎣                                                                              ↪

↪ θ₄))⋅cos(θ₅) + (-sin(θ₂)⋅cos(θ₁)⋅cos(θ₃) - sin(θ₃)⋅cos(θ₁)⋅cos(θ₂))⋅sin(θ₅)) ↪
↪                                                                              ↪
↪ θ₁))⋅cos(θ₅) + (-sin(θ₁)⋅sin(θ₂)⋅cos(θ₃) - sin(θ₁)⋅sin(θ₃)⋅cos(θ₂))⋅sin(θ₅)) ↪
↪                                                                              ↪
↪ (θ₃))⋅sin(θ₅) + (sin(θ₂)⋅cos(θ₃) + sin(θ₃)⋅cos(θ₂))⋅cos(θ₄)⋅cos(θ₅))⋅cos(θ₆) ↪
↪                          

was noch zusammengefasst werden kann

In [13]:
T06 = simplify(T06)

#display_result(T[0,6], T06)
display(T06)

⎡((sin(θ₁)⋅sin(θ₄) + cos(θ₁)⋅cos(θ₄)⋅cos(θ₂ + θ₃))⋅cos(θ₅) - sin(θ₅)⋅sin(θ₂ +  ↪
⎢                                                                              ↪
⎢((sin(θ₁)⋅cos(θ₄)⋅cos(θ₂ + θ₃) - sin(θ₄)⋅cos(θ₁))⋅cos(θ₅) - sin(θ₁)⋅sin(θ₅)⋅s ↪
⎢                                                                              ↪
⎢                                (sin(θ₅)⋅cos(θ₂ + θ₃) + sin(θ₂ + θ₃)⋅cos(θ₄)⋅ ↪
⎢                                                                              ↪
⎣                                                                              ↪

↪ θ₃)⋅cos(θ₁))⋅cos(θ₆) + (sin(θ₁)⋅cos(θ₄) - sin(θ₄)⋅cos(θ₁)⋅cos(θ₂ + θ₃))⋅sin( ↪
↪                                                                              ↪
↪ in(θ₂ + θ₃))⋅cos(θ₆) - (sin(θ₁)⋅sin(θ₄)⋅cos(θ₂ + θ₃) + cos(θ₁)⋅cos(θ₄))⋅sin( ↪
↪                                                                              ↪
↪ cos(θ₅))⋅cos(θ₆) - sin(θ₄)⋅sin(θ₆)⋅sin(θ₂ + θ₃)                              ↪
↪                          

Mithilfe dieser Matrix lässt sich nun die **Vorwärtstransformation** bestimmen 

In [ ]:
# d1=0.0892, a2=0.425, a3=0.392, d4=0.1093, d5=0.09475, d6=0.0825
def fkine(q): # q = [theta_1, ... , theta_6]
  T = T06.subs({d[0]:0.345, a[0]:0.020, a[1]:0.260, a[2]:0.020, d[3]:0.260, d[5]:0.075})
  for i in range(0, 6):
    T = T.subs({theta[i]:q[i]})
  T = T.evalf()
  return T

qz = Matrix([0, 0, 0, 0, 0, pi/2])
display('\mathbf{q}', qz)
display('\mathbf{f}(\mathbf{q})', fkine(qz))

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

# Rücktransformation mittels geometrischem Ansatz


Die **Rücktransformation** (inverses kionematisches Problem) kann entwader als analytische Lösung oder als nummerische Lösung entwickelt werden. Die **analytische** Lösung ist nur für eine bestimmte Klasse von Roboterarmen möglich. Bei Robotern mit Zentralhand kann die Aufgabenstellung in zwei Teile aufgeteit werden, die dann getrennt gelöst werden. Zuerst wird aus der Matrix 

$\newcommand{\mbf}{\mathbf}$

$$
{}^0\mbf{T}_6 \,=\,
\left( \begin{array}{cccc}
\mbf{x}_6 & \mbf{y}_6 & \mbf{z}_6 & \mbf{p}_6 \\
0 & 0 & 0 & 1
\end{array} \right) \,=\,
\left( \begin{array}{cccc}
\mbf{n} & \mbf{s} & \mbf{a} & \mbf{p}_6 \\
0 & 0 & 0 & 1
\end{array} \right) 
$$

die Position der Handwurzel der Zentralhand bestimmt:

<figure>
<center>
<img width=500 src='figures/kr6.png' />
<figcaption>Kuka KR 6 900-2</figcaption></center>
</figure>


# Rücktransformation mittels nummerischen Ansatz

Für die Rücktransformation wird die geometrische Jacobi-Matrix bestimmt.

$$
\dot{\mathbf{x}} = \mathbf{J}_\mathrm{g}(\mathbf{q}) \cdot \dot{\mathbf{q}}
$$

mit $\dot{\mathbf{x}} = (v_x, v_y, v_z, \omega_x, \omega_y, \omega_z)^\mathrm{T}$ und $\dot{\mathbf{q}} = (\dot{q}_1, \dot{q}_2, \dot{q}_3, \dot{q}_4, \dot{q}_5, \dot{q}_6)^\mathrm{T}$


Das erste Gelenk ist ein Drehgelenk, die Drehung erfolgt um die $\mathbf{z}_0$-Achse, die erste Spalte der Jacobi-Matrix $\mathbf{J}_1$ ergibt sich deshalb wie folgt:
$$ 
\mathbf{J}_1^\mathrm{R} = 
\begin{pmatrix}
	\mathbf{J}_\mathrm{v}\\
	\mathbf{J}_\omega
\end{pmatrix}
= 
\begin{pmatrix}
	\mathbf{z}_0 \times (\mathbf{p}_{6} - \mathbf{p}_{0})\\
	\mathbf{z}_{0}
\end{pmatrix}
$$

mit

$$
{}^0\mathbf{T}_{0} = 
\begin{pmatrix}
1 & 0 & 0 & 0 \\
0 & 1 & 0 & 0 \\
0 & 0 & 1 & 0  \\
0 & 0 & 0 & 1
\end{pmatrix}
= 
\begin{pmatrix}
\mathbf{x}_{0} & \mathbf{y}_{0} & \mathbf{z}_{0} & \mathbf{p}_{0} \\
0 & 0 & 0 & 1
\end{pmatrix}
$$

und 

$$
{}^0\mathbf{T}_{6} = {}^0\mathbf{T}_{1} \cdot {}^1\mathbf{T}_{2} \cdot 
{}^2\mathbf{T}_{3} \cdot {}^3\mathbf{T}_{4} \cdot {}^4\mathbf{T}_{5} \cdot 
{}^5\mathbf{T}_{6} = 
\begin{pmatrix}
\mathbf{x}_{6} & \mathbf{y}_{6} & \mathbf{z}_{6} & \mathbf{p}_{6} \\
0 & 0 & 0 & 1
\end{pmatrix}
$$


In [14]:
J = symbols('\mathbf{J}_1:7')
Jgg = symbols('\mathbf{J}_g')
Jgs = Matrix(1,6,[J[0], J[1], J[2], J[3], J[4],J[5]])

display_result(Jgg, Jgs)

<IPython.core.display.Latex object>

In [15]:
T00 = eye(4)     # Transformation von K0 nach K0 ist I4x4
z0 = T00[:3,2]
p0 = T00[:3,3]
p6 = T06[:3,3]
Jg = zeros(6,6)
Jg[:,0] =  simplify(Matrix([z0.cross(p6 - p0), z0]))

display_result(J[0], Jg[:,0])

<IPython.core.display.Latex object>

Das Ergebnis von $\mathbf{J}_1$ zeigt, dass bei Bewegung des ersten Gelenks ($\theta_1$), sich die Position in der z-Komponente nicht ändert und dass die Orientierung sich nur um die z-Achse verändert.


In [16]:

#Die weiteren fünf Achsen werden analog dazu berechnet:

Tm = [T00, T01, T12, T23, T34, T45]
T0i = Tm[0]
for i in range(1, 6):
  T0i = T0i * Tm[i]
  T0i = simplify(T0i)
  z_i = T0i[:3,2]
  p_i = T0i[:3,3]
  Jg[:,i] =  simplify(Matrix([z_i.cross(p6 - p_i), z_i]))

  display_result(J[i], Jg[:,i])

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

Bei Bewegung des letzten Gelenks wid die Position nicht mehr geändert.

Zum Schluss geben wir die nun vollständige Jacobi-Matrix $\mathbf{J}_g$ aus:

In [17]:
display_result(Jgg, Jgs)
display_result(Jg)

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

---
Mithilfe der Jacobi-Matrix lässt sich nun die Rücktransformation numerisch bestimmen. Ausgehend von einem Startpunkt der Gelenkwinkel $\mathbf{q}_k$ kann die Lösung für $\mathbf{q} = \mathbf{f}^{-1}(\mathbf{x})$ iterativ mittels Gauß-Newton-Verfahren bestimmt werden:
$$
\mathbf{q}_{k+1} = \mathbf{q}_k + \mathbf{J}^{-1}(\mathbf{q}_k) \cdot (\mathbf{x} - \mathbf{x}_k) \quad \text{mit} \quad \mathbf{x}_k = \mathbf{f}(\mathbf{q}_k)
$$

Sobald eine hinreichend genaue Lösung ($\left| \mathbf{x} - \mathbf{x}_k \right| < \epsilon$) für $\mathbf{x}$ gefunden ist, wird die Iteration abgebrochen. 


In [18]:
def mod_2pi(a): # -pi < a < pi
  a = a % (2*pi)
  if a < -pi:
    a += 2*pi 
  if a > pi:
    a -= 2*pi
  return a 

def tr2delta(T0, T1): # Bewegungsdelta von T0 -> T1, siehe auch tr2delta aus Robotics Toolbox Peter Corke
  p = T1[:3, 3]    # Zielposition
  R = T1[:3, :3]   # Zielorientierung als Rotationsmatrix
  p0 = T0[:3,3]    # Ausgangs-Position in T0
  R0 = T0[:3,:3]   # Ausgangs-Orientierung in T0 
  dp = (p - p0)    # Positionsdelta
  dR = R * R0.transpose()
  do = 0.5 * Matrix([dR[2,1] - dR[1,2], dR[0,2] - dR[2,0], dR[1,0] - dR[0,1]]) # Orientierungsdelta
  # do = 0.5 * (R0[:,0].cross(R[:,0]) + R0[:,1].cross(R[:,1]) + R0[:,2].cross(R[:,2])) 
  dx = Matrix([dp, do]).evalf() # Delta in der Pose   
  return dx

def ikine(T, qk): # T homogene Transformationsmatrix als Ziel 
  Js = Jg.subs({d[0]:0.345, a[0]:0.020, a[1]:0.260, a[2]:0.020, d[3]:0.260, d[5]:0.075})
  for k in range(0, 30):
    Tk = fkine(qk)
    dx = tr2delta(Tk, T)
    norm_x = sqrt(dx.dot(dx))
    if norm_x < 0.000001:
      print('\n Abbruch bei k = ' + str(k) + '\n')
      for j in range(0, 6):  
        qk[j] = mod_2pi(qk[j]) # -pi < theta_i < pi
      break
    Jk = Js
    for i in range(0, 6):
      Jk = Jk.subs({theta[i]:qk[i]})
    Jkinv = (Jk.evalf()).inv()
    qk = qk + Jkinv * dx
    qk = qk.evalf()
    #print(qk)
  return qk.evalf()  

#
q = Matrix([-pi/2, 0, 0, pi/2, 0, -pi/2])
display_latex_result('\mathbf{q}', q)
qk = Matrix([-0.5, -0.5, -0.5, -0.5, -0.1, -0.5])
T = fkine(q)
display_latex_result('\mathbf{T} = \mathbf{f}(\mathbf{q})', T)
ql = ikine(T, qk)
display_latex_result('\mathbf{q} = \mathbf{f}^{-1}(\mathbf{T})', ql.evalf(6))
Tn = fkine(ql)
display_latex_result('\mathbf{T} = \mathbf{f}(\mathbf{q})',Tn.evalf(6))

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>


 Abbruch bei k = 17



<IPython.core.display.Latex object>

<IPython.core.display.Latex object>